In [0]:
base_path = "/Volumes/de_workspace26/bronze_aditya_sah_assessment/raw_data/"

## Reading CSV files
 

In [0]:
orders_df = spark.read.csv(
    base_path + "orders/",
    header=True,
    inferSchema=True
)

customers_df = spark.read.csv(
    base_path + "customers/",
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    base_path + "products/",
    header=True,
    inferSchema=True
)

## Adding metadata columns


In [0]:
from pyspark.sql.functions import current_timestamp, lit

orders_df = orders_df.withColumn("_ingested_at", current_timestamp()) \
                     .withColumn("_source_file", lit("orders_batch"))

customers_df = customers_df.withColumn("_ingested_at", current_timestamp()) \
                           .withColumn("_source_file", lit("customers_batch"))

products_df = products_df.withColumn("_ingested_at", current_timestamp()) \
                         .withColumn("_source_file", lit("products_batch"))

## Writing to bronze delta table

In [0]:
orders_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("de_workspace26.bronze_aditya_sah_assessment.orders")

customers_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("de_workspace26.bronze_aditya_sah_assessment.customers")

products_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("de_workspace26.bronze_aditya_sah_assessment.products")

### Adding constraint

In [0]:
%sql
ALTER TABLE de_workspace26.bronze_aditya_sah_assessment.orders DROP CONSTRAINT order_id_not_null;
ALTER TABLE de_workspace26.bronze_aditya_sah_assessment.orders
ADD CONSTRAINT order_id_not_null CHECK (order_id IS NOT NULL);

In [0]:
%sql
DESCRIBE HISTORY de_workspace26.bronze_aditya_sah_assessment.orders;

## Auto Loader

### Create Stream Ingestion

In [0]:
orders_stream_df = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option(
        "cloudFiles.schemaLocation",
        base_path + "_schema/orders"
    ) \
    .load(base_path + "orders/")

### Adding metadata

In [0]:
orders_stream_df = orders_stream_df.withColumn("_ingested_at", current_timestamp()) \
                                   .withColumn("_source_file", lit("orders_stream"))

In [0]:
%sql
VACUUM de_workspace26.silver_aditya_sah_assessment.orders RETAIN 168 HOURS;